In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
# Step 1 — Enable Change Data Feed on the embeddings table
spark.sql("""
    ALTER TABLE uc13.ingestion.embeddings 
    SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")
print("Change Data Feed enabled")

In [0]:
# Step 2 — Create index
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.vectorsearch import (
    VectorIndexType,
    DeltaSyncVectorIndexSpecRequest,
    EmbeddingVectorColumn,
    PipelineType
)

w = WorkspaceClient()

index = w.vector_search_indexes.create_index(
    name="uc13.ingestion.embeddings_index",
    endpoint_name="uc13-vector-search",
    primary_key="chunk_id",
    index_type=VectorIndexType.DELTA_SYNC,
    delta_sync_index_spec=DeltaSyncVectorIndexSpecRequest(
        source_table="uc13.ingestion.embeddings",
        pipeline_type=PipelineType.TRIGGERED,
        embedding_vector_columns=[
            EmbeddingVectorColumn(
                name="embedding",
                embedding_dimension=1024
            )
        ],
        columns_to_sync=[
            "chunk_id", "doc_id", "file_name",
            "workstream", "priority_tier",
            "company_name", "source_type",
        ]
    )
)

print(f"Index created: {index.name}")
print(f"Status: {index.status}")

In [0]:
index_info = w.vector_search_indexes.get_index(
    index_name="uc13.ingestion.embeddings_index"
)
print(f"Ready: {index_info.status.ready}")
print(f"Message: {index_info.status.message}")
print(f"Indexed rows: {index_info.status.indexed_row_count}")

##### This should be erased:

In [0]:
import os
import sys
from pathlib import Path

def get_current_path():
    try:
        notebook_path = (
            dbutils.notebook.entry_point
            .getDbutils()
            .notebook()
            .getContext()
            .notebookPath()
            .get()
        )
        return Path("/Workspace") / notebook_path.lstrip("/")
    except Exception:
        return Path(os.getcwd())

def find_repo_root(marker="agents"):
    current_path = get_current_path()

    if current_path.is_file():
        current_path = current_path.parent

    for path in [current_path, *current_path.parents]:
        if (path / marker).exists():
            return str(path)

    raise RuntimeError(f"No se encontró una carpeta padre que contenga '{marker}'")

repo_root = find_repo_root()

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print("repo_root:", repo_root)

from agents.shared.retrieval import semantic_search

print("Import OK")

results = semantic_search(
    query="company business description services industry vertical",
    spark=spark,
    top_k=5
)

print(f"Results: {len(results)}")
for r in results:
    print(f"\n[{r.file_name}] {r.section_header}")
    print(f"{r.chunk_text[:300]}")

In [0]:
# Test with CIM-specific query and tier 1 filter
results = semantic_search(
    query="Elder Care homecare services home care caregiver New York New Jersey",
    spark=spark,
    top_k=5,
    tier_filter=1
)

print(f"Results: {len(results)}")
for r in results:
    print(f"\n[{r.file_name} | T{r.priority_tier} | {r.workstream}]")
    print(f"Section: {r.section_header}")
    print(f"{r.chunk_text[:400]}")
    print("---")

In [0]:
%sql
SELECT DISTINCT c.file_name, r.workstream, r.priority_tier, COUNT(*) as chunks
FROM uc13.ingestion.chunks c
JOIN uc13.classification.doc_relevance r ON c.file_name = r.file_name
WHERE r.priority_tier = 1
GROUP BY c.file_name, r.workstream, r.priority_tier
ORDER BY r.workstream, chunks DESC

In [0]:
%sql
SELECT * FROM uc13.ingestion.embeddings LIMIT 10;